In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
all_az_permits = pd.read_csv("../data/permitpower-az.csv", escapechar="\\")

In [ ]:
top100 = all_az_permits.head(100)

# Basic shape
print(f"Columns ({len(top100.columns)}): {list(top100.columns)}\n")

# Status breakdown
print("--- STATUS ---")
print(top100['STATUS'].value_counts(), "\n")

# Permit type breakdown
print("--- TYPE ---")
print(top100['TYPE'].value_counts(), "\n")

# Cities
print("--- CITY ---")
print(top100['CITY'].value_counts().head(10), "\n")

# Date range
for col in ['FILE_DATE', 'ISSUE_DATE', 'FINAL_DATE']:
    if col in top100.columns:
        print(f"{col}: {top100[col].min()} → {top100[col].max()}")

top100

In [ ]:
len(all_az_permits)

In [ ]:
sample = all_az_permits.sample(n=10_000, random_state=42)
sample.to_csv("../data/az_permits_sample_10k.csv", index=False)
print(f"Wrote {len(sample)} rows to ../data/az_permits_sample_10k.csv")

In [ ]:
final_permits = all_az_permits[all_az_permits['STATUS'] == 'final']
print(f"Final permits: {len(final_permits):,}")
print(f"Distinct jurisdictions: {final_permits['JURISDICTION'].nunique()}")

In [ ]:
dur = pd.to_numeric(final_permits['APPROVAL_DURATION'], errors='coerce').dropna()

q25 = dur.quantile(0.25)
q75 = dur.quantile(0.75)

print(f"Records with numeric APPROVAL_DURATION: {len(dur):,} of {len(final_permits):,}")
print(f"Median:  {dur.median():,.1f}")
print(f"Mean:    {dur.mean():,.1f}")
print(f"Mean (bottom 25%, <= {q25:.0f}): {dur[dur <= q25].mean():,.1f}")
print(f"Mean (top 25%,    >= {q75:.0f}): {dur[dur >= q75].mean():,.1f}")

In [ ]:
pima = final_permits[final_permits['COUNTY'] == 'PIMA'].copy()
pima['FINAL_YEAR'] = pd.to_datetime(pima['FINAL_DATE'], errors='coerce').dt.year
pima['APPROVAL_DURATION'] = pd.to_numeric(pima['APPROVAL_DURATION'], errors='coerce')
pima['TOTAL_DURATION'] = pd.to_numeric(pima['TOTAL_DURATION'], errors='coerce')

pima.groupby('FINAL_YEAR')[['APPROVAL_DURATION', 'TOTAL_DURATION']].mean().round(1)

In [ ]:
# Regression: TOTAL_DURATION ~ APPROVAL_DURATION (all final permits)
reg_data = final_permits[['APPROVAL_DURATION', 'TOTAL_DURATION']].copy()
reg_data['APPROVAL_DURATION'] = pd.to_numeric(reg_data['APPROVAL_DURATION'], errors='coerce')
reg_data['TOTAL_DURATION'] = pd.to_numeric(reg_data['TOTAL_DURATION'], errors='coerce')
reg_data = reg_data.dropna()

slope, intercept, r, p, se = stats.linregress(reg_data['APPROVAL_DURATION'], reg_data['TOTAL_DURATION'])

print(f"Slope (days of total per permit day): {slope:.2f}")
print(f"Intercept:                            {intercept:.1f}")
print(f"R²:                                   {r**2:.3f}")
print(f"p-value:                              {p:.2e}")
print(f"n:                                    {len(reg_data):,}")

In [ ]:
# Check the approval/total ratio different ways
ratio = reg_data['APPROVAL_DURATION'] / reg_data['TOTAL_DURATION']
print(f"Mean approval/total ratio:   {ratio.mean():.2%}")
print(f"Median approval/total ratio: {ratio.median():.2%}")
print()
print(f"Mean approval:  {reg_data['APPROVAL_DURATION'].mean():.1f} days")
print(f"Mean total:     {reg_data['TOTAL_DURATION'].mean():.1f} days")
print(f"Median approval:  {reg_data['APPROVAL_DURATION'].median():.1f} days")
print(f"Median total:     {reg_data['TOTAL_DURATION'].median():.1f} days")
print()
# Simple ratio method: 10% * (approval/total)
print(f"Simple method (10% × median ratio): {0.10 * ratio.median():.1%}")

In [ ]:
approval = pd.to_numeric(final_permits['APPROVAL_DURATION'], errors='coerce')
juris_median = (
    final_permits.assign(APPROVAL_DURATION=approval)
    .groupby('JURISDICTION')['APPROVAL_DURATION']
    .median()
    .dropna()
)

juris_median = juris_median[juris_median > 0]

q25 = juris_median.quantile(0.25)
q75 = juris_median.quantile(0.75)

print(f"Jurisdictions analyzed: {len(juris_median):,}")
print(f"Fastest 25% (median <= {q25:.0f} days): median = {juris_median[juris_median <= q25].median():.1f} days")
print(f"Slowest 25% (median >= {q75:.0f} days): median = {juris_median[juris_median >= q75].median():.1f} days")
print()
print("Fastest jurisdictions:")
print(juris_median.sort_values().head(10).round(1))
print("\nSlowest jurisdictions:")
print(juris_median.sort_values().tail(10).round(1))

In [ ]:
# Summary statistics
all_az_permits.describe(include='all')

In [ ]:
# Null counts
null_counts = all_az_permits.isnull().sum()
null_counts[null_counts > 0].sort_values(ascending=False)

In [ ]:
# Value counts for categorical columns
cat_cols = all_az_permits.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(all_az_permits[col].value_counts().head(20))